# Machine Learning Project: Hospital Readmission Prediction

## 1. Problem Framing
* **Stakeholder:** Hospital Care-Planning Team and Administrators.
* **Decision Need:** Identifying which diabetic patients require immediate extra follow-up care and resource allocation after discharge to prevent early readmission.
* **Primary Lens:** 30-day hospital readmission risk prediction.
* **Secondary Lens:** High-risk patient profiling (identifying common characteristics and risk drivers).
* **Unit of Analysis:** One single hospital encounter (visit) by a diabetic patient.
* **Exact Task/Output:** Supervised multi-class classification to predict whether a patient will be readmitted within 30 days, after 30 days, or not at all (3 categories: `<30`, `>30`, `NO`).
* **Rationale:** By predicting immediate-risk patients (<30 days) versus other cases, the hospital can efficiently allocate limited post-discharge resources to the most critical cases.



## Exploratory Data Analysis (EDA)
This notebook performs a comprehensive EDA following the project requirements. We apply analysis across ALL features to ensure no potential signal is missed.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import math
import os

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style="whitegrid")

# Create directory for plots
os.makedirs('plots/EDA', exist_ok=True)



## 1. Dataset Overview
We begin by loading the dataset, checking its shape, data types, and identifying the key identifier columns vs features.


In [ ]:
# keep_default_na=False -> text "None" is NOT turned into NaN. IDs stay integers.
df = pd.read_csv('dataset_extracted/diabetic_data.csv', keep_default_na=False)

# Parse IDS_mapping.csv (3 tables stacked, blank-row separated)
with open('dataset_extracted/IDS_mapping.csv', 'r') as f:
    lines = f.readlines()

mapping_dicts, current_map = {}, None
for line in lines:
    line = line.strip()
    if not line or line.strip(',') == '':          # skip blank and ",," separator rows
        continue
    if line.endswith('_id,description'):
        current_map = line.split(',')[0]
        mapping_dicts[current_map] = {}
    elif current_map:
        key, val = line.split(',', 1)
        mapping_dicts[current_map][int(key)] = val.strip().strip('"')   # int keys

# Readable *_desc columns (original numeric codes stay unchanged)
code_cols = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
desc_cols = []
for c in code_cols:
    new = c.replace('_id', '_desc')
    # Map the integers, but if the dictionary doesn't have the integer, fill the resulting NaN with 'Unknown/Unmapped'
    df[new] = df[c].map(mapping_dicts[c]).fillna('Unknown/Unmapped')
    desc_cols.append(new)

print(f"Unmapped codes (now filled with 'Unknown/Unmapped'): {(df[desc_cols] == 'Unknown/Unmapped').sum().to_dict()}")
print(f"Dataset Shape: {df.shape}")
print("\nMemory Usage:")
df.info(memory_usage='deep')

# Separate columns by type
id_cols = ['encounter_id', 'patient_nbr']
target_col = 'readmitted'
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(exclude=['object']).columns.tolist()

categorical_cols = [c for c in categorical_cols if c not in id_cols + [target_col] + desc_cols]
numeric_cols = [c for c in numeric_cols if c not in id_cols + [target_col] + code_cols]
categorical_cols += code_cols          # codes are categories, not quantities

print(f"\nIdentifier Columns: {id_cols}")
print(f"Target Column: {target_col}")
print(f"Numerical Columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical Columns ({len(categorical_cols)}): {categorical_cols}")

# Basic first look
from IPython.display import display
print("\n--- Head ---");  display(df.head())
print("\n--- Tail ---");  display(df.tail())
print("\n--- Numeric Describe ---");     display(df[numeric_cols].describe().T)
print("\n--- Categorical Describe ---");  display(df[categorical_cols].describe().T)
print("\n--- Nunique ---");  display(df.nunique().sort_values())



### What we learned
- The dataset has 101,766 rows and 50 columns.
- `encounter_id` and `patient_nbr` are identifiers, and `readmitted` is the target.
- Most features are stored as `object` (categorical/strings), and there are several numerical features.
- Missing values are coded as `?` instead of actual NaNs.

### What we will do about it
- We will convert the `?` placeholders into proper `NaN` values.
- We will dynamically loop through all identified numerical and categorical columns for comprehensive EDA rather than selectively picking a few.



## 2. Target Analysis
The target variable is `readmitted` with 3 classes: `NO`, `>30`, and `<30`.


In [ ]:
# We will use the 3 categories as our target
df['target'] = df['readmitted']

# Class counts and percentages
counts = df['target'].value_counts()
pcts = df['target'].value_counts(normalize=True) * 100
target_stats = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
print(target_stats)

# Plot
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='target', order=['NO', '>30', '<30'])
plt.title('Distribution of Readmission (3 Classes)')
plt.xlabel('Readmitted Category')
plt.ylabel('Count')

# Annotate counts
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.savefig('plots/EDA/target_distribution.png', bbox_inches='tight')
plt.show()



### What we learned
- The dataset is distributed across 3 classes: NO (~54%), >30 (~35%), and <30 (~11%).
- The class <30 (early readmission) is the minority class.

### What we will do about it
- We need to handle this as a multi-class classification problem predicting exactly these 3 categories.
- We must handle class imbalance during modeling since the `<30` class is much smaller than `NO` and `>30`.



## 3. Missing Values
We replace all string placeholders with NaN and calculate missingness across ALL columns.


In [ ]:
# Replace placeholders with NaN
placeholders = ['?', 'Unknown/Invalid', 'Not Available', 'NULL', 'Not Mapped']
df.replace(placeholders, np.nan, inplace=True)

# Calculate missing percentages for all features
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Missing_Pct': missing_pct})
missing_df = missing_df[missing_df['Missing_Pct'] > 0].sort_values(by='Missing_Pct', ascending=False)

print(missing_df)

# Plot
plt.figure(figsize=(10, 5))
if not missing_df.empty:
    ax = sns.barplot(x=missing_df.index, y='Missing_Pct', data=missing_df, palette='viridis')
    for c in ax.containers: ax.bar_label(c, fmt='%.1f%%', fontsize=9)
    plt.xticks(rotation=45, ha='right')
    plt.title('Percentage of Missing Values per Feature (Only features with missing data)')
    plt.ylabel('% Missing')
    plt.savefig('plots/EDA/missing_values.png', bbox_inches='tight')
    plt.show()
else:
    print("No missing values found after processing!")



### What we learned
- `weight` is missing in ~97% of the records.
- `medical_specialty` is missing ~49%.
- `payer_code` is missing ~39%.
- `race` and a few others have <5% missing values.

### What we will do about it
- **Drop** `weight` entirely due to excessive missingness.
- For `medical_specialty` and `payer_code`, we will fill with an `"Unknown"` category because the fact that it is missing might itself carry predictive information (e.g., lack of specialist access).
- For low-missingness columns like `race`, we will impute with the mode (most frequent) or drop those specific rows.



## 4. Duplicates & Patient Leakage Risk
Checking for identical rows and repeated encounters for the same patient.


In [ ]:
# Exact duplicate rows
print(f"Exact duplicate rows: {df.duplicated().sum()}")

# Patient encounters distribution
unique_patients = df['patient_nbr'].nunique()
print(f"Unique patients: {unique_patients} across {len(df)} total encounters.")

encounter_counts = df['patient_nbr'].value_counts()
print(f"Max encounters for a single patient: {encounter_counts.max()}")

# Group encounters into 1, 2, ..., 10, 11+
grouped_encounters = encounter_counts.apply(lambda x: str(x) if x <= 10 else '11+')
order = [str(i) for i in range(1, 11)] + ['11+']
grouped_counts = grouped_encounters.value_counts().reindex(order).fillna(0)

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=grouped_counts.index, y=grouped_counts.values, palette='viridis')
plt.title('Number of Encounters per Patient')
plt.xlabel('Number of Encounters')
plt.ylabel('Number of Patients')

# Annotate counts above bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=9, xytext=(0, 4), textcoords='offset points')

plt.savefig('plots/EDA/encounters_per_patient.png', bbox_inches='tight')
plt.show()



### What we learned
- There are no exact duplicate rows.
- However, there are ~71,518 unique patients for ~101,766 encounters, meaning tens of thousands of patients visit multiple times (up to 40 times!).

### What we will do about it
- We **must split the train/test sets by `patient_nbr`**, not randomly. Random splitting would cause data leakage. 
- We will likely extract only the first encounter per patient during preprocessing, or use GroupKFold.



## 5. Univariate Analysis
Let's explore the distribution of **ALL** numerical and categorical variables dynamically. We will plot every feature.


In [ ]:
# Plot ALL Numerical features
num_cols_count = len(numeric_cols)
rows = math.ceil(num_cols_count / 3)
fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 4))
axes = axes.flatten()

num_tables = {}
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='teal', bins=30)
    axes[i].set_title(f'Distribution: {col}')

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('plots/EDA/univariate_numerical.png', bbox_inches='tight')
plt.show()

# Stats
stats_df = df[numeric_cols].describe().T
stats_df['skewness'] = df[numeric_cols].skew()

# IQR Outlier Detection
outliers_list = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_count = ((df[col] < lower) | (df[col] > upper)).sum()
    outliers_list.append(outliers_count)
    
stats_df['outliers (IQR)'] = outliers_list
print("\nNumerical Statistics & Outliers:")
print(stats_df)



### Outlier Visualization (Boxplots)
To visually inspect the outliers detected by the IQR method above, we plot boxplots for all numerical features.


In [ ]:
# Boxplots for numerical outliers
fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 4))
axes = axes.flatten()

num_tables = {}
for i, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[i], color='coral')
    axes[i].set_title(f'Outliers: {col}')

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('plots/EDA/outliers_boxplots.png', bbox_inches='tight')
plt.show()



### What we learned (Numerical Distributions & Outliers)
- Most numerical features, particularly count-based variables like `number_inpatient`, `number_emergency`, and `number_outpatient`, are heavily right-skewed and zero-inflated.
- The boxplots reveal severe right-tail outliers across almost all numerical features. This reflects that a small subset of "high-utilizer" patients have extremely high historical hospital usage.
- `time_in_hospital` and `num_lab_procedures` follow more normalized, albeit slightly skewed, distributions.

### What we will do about it
- We will NOT simply drop these outliers, as these extreme "high-utilizer" patients carry highly valuable predictive signals for readmission.
- Instead, due to the extreme right-skew and heavy presence of zeros, we will apply a `log1p` (`log(1 + x)`) transformation to these features during preprocessing.
- `log1p` safely compresses the extreme outlier tails and handles the zeros gracefully, which will significantly stabilize linear models without losing the underlying signal.



In [ ]:
# Plot ALL Categorical features
# Note: For high cardinality features like diagnoses (diag_1, diag_2, diag_3), we only plot the top 15 categories to keep it readable.
cat_cols_count = len(categorical_cols)
rows = math.ceil(cat_cols_count / 3)
fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 4))
axes = axes.flatten()

cat_tables = {}
for i, col in enumerate(categorical_cols):
    # Get top 15 categories if there are too many unique values
    top_cats = df[col].value_counts().nlargest(15).index
    sns.countplot(data=df[df[col].isin(top_cats)], x=col, ax=axes[i], palette='Set2', order=top_cats)
    for c in axes[i].containers: axes[i].bar_label(c, fmt='%.0f', fontsize=8)
    axes[i].set_title(f'Count: {col} (Top 15)')
    axes[i].tick_params(axis='x', rotation=45)

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('plots/EDA/univariate_categorical.png', bbox_inches='tight')
plt.show()



### What we learned
- Count-based numerical variables (`number_emergency`, `number_inpatient`) are heavily right-skewed with most values at 0.
- `time_in_hospital` and `num_lab_procedures` follow somewhat skewed normal distributions.
- Dozens of medication columns (like `chlorpropamide`, `tolbutamide`) are overwhelmingly heavily imbalanced, where >99% of patients have "No" (not prescribed).
- The ICD-9 diagnosis codes (`diag_1`, `diag_2`, `diag_3`) have hundreds of unique values.

### What we will do about it
- We will consider applying log transformations to highly skewed numerical features.
- We should drop medication features that have close to zero variance (e.g., 99.9% "No") as they offer no predictive power.
- Diagnosis codes must be grouped (e.g., grouping by disease category like Circulatory, Respiratory) rather than one-hot encoding hundreds of unique codes.



## 6. Feature vs Target (Readmission Risk)
We analyze how **ALL** features correlate with the 3 target classes using stacked bar charts.


### Categorical Features vs Target


In [ ]:
import scipy.stats as stats
from IPython.display import display

# 1. Compute Chi-Square and Cramer's V for all categorical features
chi2_results = []
for col in categorical_cols:
    contingency = pd.crosstab(df[col], df['target'])
    chi2, p, dof, ex = stats.chi2_contingency(contingency)
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    cramer_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    chi2_results.append({'Feature': col, 'Chi2': chi2, 'p-value': p, 'Cramers_V': cramer_v})

chi2_df = pd.DataFrame(chi2_results).sort_values(by='Cramers_V', ascending=False)
print("\n--- Categorical Features Ranked by Cramér's V (Association with Target) ---")
display(chi2_df.head(15))

# Plot top 9 most associated features based on Cramer's V
top_features = chi2_df['Feature'].head(9).tolist()
if 'age' not in top_features:
    top_features[-1] = 'age' # Ensure age is included to demonstrate correct sorting

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

cat_tables = {}
for i, col in enumerate(top_features):
    ax = axes[i]
    
    # If a feature has massive cardinality (e.g., diag codes), group the tail for plotting only
    plot_col = df[col]
    if plot_col.nunique() > 20:
        top_cats = plot_col.value_counts().nlargest(19).index
        plot_col = plot_col.apply(lambda x: x if x in top_cats else 'Other')
        
    ct = pd.crosstab(plot_col, df['target'], normalize='index') * 100
    
    # Sort by risk, EXCEPT for age which must be in natural order
    if col == 'age':
        ct = ct.sort_index()
    elif '<30' in ct.columns:
        ct = ct.sort_values(by='<30', ascending=False)
        
    ct[['NO', '>30', '<30']].plot(kind='bar', stacked=True, ax=ax, colormap='viridis')
    ax.set_title(f'Readmission by {col}')
    ax.set_ylabel('% of Patients')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Target')
    
    cat_tables[col] = ct

plt.tight_layout()
plt.savefig('plots/EDA/feature_vs_target_categorical.png', bbox_inches='tight')
plt.show()

print("\n--- Categorical Feature vs Target (<30 Days Risk Tables) ---")
for col, ct in cat_tables.items():
    if '<30' in ct.columns:
        display(pd.DataFrame({f'{col}': ct.index[:5], '<30 Risk %': ct['<30'].values[:5].round(2)}))



### Numerical Features vs Target


In [ ]:
# Plot for ALL numerical features (by binning them)
num_rows = math.ceil(num_cols_count / 3)
fig, axes = plt.subplots(num_rows, 3, figsize=(18, num_rows * 4))
axes = axes.flatten()

num_tables = {}
for i, col in enumerate(numeric_cols):
    # Create 4 quantile bins for the numerical feature
    bin_col = f'{col}_bin'
    try:
        df[bin_col] = pd.qcut(df[col], q=4, duplicates='drop')
        ct = plot_readmission_distribution(bin_col, axes[i])
        if ct is not None:
            num_tables[bin_col] = ct
    except ValueError:
        # If qcut fails (e.g. too many zeros), use regular cut
        df[bin_col] = pd.cut(df[col], bins=4)
        ct = plot_readmission_distribution(bin_col, axes[i])
        if ct is not None:
            num_tables[bin_col] = ct

# Hide any empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('plots/EDA/feature_vs_target_numerical.png', bbox_inches='tight')
plt.show()

from IPython.display import display
print("\n--- Numerical Feature vs Target (<30 Days Risk Tables) ---")
for col, ct in num_tables.items():
    if '<30' in ct.columns:
        display(pd.DataFrame({f'{col} Bins (Highest Risk First)': ct.index[:4].astype(str), '<30 Risk %': ct['<30'].values[:4].round(2)}))



### What we learned
- Across almost all features, the proportion of `<30` and `>30` fluctuates.
- Certain `discharge_disposition_id` codes have very high readmission rates.
- Patients with higher `number_inpatient` (previous inpatient visits) have noticeably higher proportions of early readmission (`<30`).
- Medication changes (`change` = "Ch") slightly increase the likelihood of readmission.

### What we will do about it
- `number_inpatient` will likely be a very strong predictive feature.
- We will group similar discharge disposition codes and admission source codes to strengthen the signal for the models.



## 7. Feature Relationships
Checking for multicollinearity amongst ALL numerical features.


In [ ]:
# Correlation heatmap
corr = df[numeric_cols].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', cbar=True, square=True)
plt.title('Spearman Correlation between ALL Numerical Features')
plt.savefig('plots/EDA/correlation_heatmap.png', bbox_inches='tight')
plt.show()



### What we learned
- `num_medications` and `time_in_hospital` are positively correlated (0.47).
- `num_medications` and `num_procedures` also have positive correlation (0.39).
- Overall, there is no severe multicollinearity (no correlations > 0.8).

### What we will do about it
- We don't need to aggressively drop numerical features due to multicollinearity. All of them can be retained for the tree-based models.



## 8. Data Quality & Conceptual Leakage
Certain patients (e.g., deceased or transferred to hospice) conceptually cannot be readmitted. Including them can bias the model.


In [ ]:
# Check discharge_disposition_id for expired/hospice patients
# Codes 11, 13, 14, 19, 20, 21 represent death or hospice
terminal_codes = [11, 13, 14, 19, 20, 21]
terminal_patients = df[df['discharge_disposition_id'].isin(terminal_codes)]

print(f"Patients discharged to hospice or expired: {len(terminal_patients)}")



### What we learned
- Thousands of patients were discharged to hospice or expired.
- These patients are virtually guaranteed not to be readmitted, meaning they are "easy" negatives that artificially inflate model performance for the 'NO' class.

### What we will do about it
- We will **drop** all rows with these discharge disposition codes during preprocessing to ensure the model focuses on patients who actually *could* be readmitted.



## 9. Targeted EDA (Before Preprocessing)
We need to explicitly analyze the ID mappings and the internal distributions of test results and medications before we can safely finalize Phase 4 and Phase 5.


In [ ]:
# A. ID Codes Distributions
id_cols_to_check = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(id_cols_to_check):
    counts = df[col].value_counts().nlargest(10)
    sns.barplot(x=counts.index, y=counts.values, ax=axes[i], order=counts.index, palette='magma')
    for c in axes[i].containers: axes[i].bar_label(c, fmt='%.0f', fontsize=9)
    axes[i].set_title(f'Top 10: {col}')
    
plt.tight_layout()
plt.savefig('plots/EDA/id_mappings_targeted.png', bbox_inches='tight')
plt.show()



In [ ]:
# B. Test Results & Medications Ordinality Check
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot A1Cresult vs Readmission
plot_readmission_distribution('A1Cresult', axes[0])
axes[0].set_title("A1Cresult vs Target")

# Plot Insulin (since it's the most common) vs Readmission
plot_readmission_distribution('insulin', axes[1])
axes[1].set_title("Insulin vs Target")

plt.tight_layout()
plt.savefig('plots/EDA/meds_tests_targeted.png', bbox_inches='tight')
plt.show()



### What we learned from Targeted EDA
- For `A1Cresult`, the proportion of early readmissions (`<30`) does **not** strictly increase from `Norm` -> `>7` -> `>8`. This means it is NOT strictly ordinal in relation to our target.
- For `insulin`, the `Up` and `Down` categories exhibit different risk profiles compared to `Steady`.
- ID columns are heavily concentrated in just 2-4 categories (e.g., ID 1, 2, 3), meaning we can safely group the long tail into "Other".

### What we will do about it
- We will group the ID columns based on their Top 3 mapped categories and map the rest to "Other" in Phase 4.
- We will use **One-Hot Encoding** (Nominal) for `A1Cresult` and medications in Phase 5 rather than Ordinal Encoding, as it prevents forcing false linear assumptions on non-linear risks.



## 10. EDA Conclusions & Next Steps
We have a clear path forward for the Preprocessing phase.

| Finding | Implication | Action |
| :--- | :--- | :--- |
| `weight` has 97% missing data | Unusable feature | Drop column |
| Near-zero variance on many medications | No predictive power | Drop columns with >99% 'No' |
| Multi-class imbalance | Accuracy is a bad metric | Use SMOTE/class weights |
| High encounters per patient | Random split causes leakage | GroupKFold / Split by `patient_nbr` |
| Hospice/Death discharge codes | Inflates negative classes | Drop rows with terminal codes |
| Diagnosis codes have 700+ unique values | High dimensionality | Group ICD-9 codes by category |
| Tests/Meds lack strict monotonic correlation | Ordinal encoding creates false math | Use One-Hot Encoding |



# Preprocessing

The following phases implement a rigorous, leakage-free preprocessing and feature engineering pipeline. Each step is documented with a clear rationale and its outcome. The pipeline strictly follows best practices: all data-dependent transformations (encoding, scaling) are **fit on the training set only** and applied to both sets.


## Preprocessing Step 1 · Load Data & Understand Its Structure
We confirm the dataset shape, column types, and data presence one final time before making any permanent changes.


In [ ]:
print(f"Dataset shape before any preprocessing: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nData types:")
print(df.dtypes.value_counts())
print("\nNull counts (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))



**What we confirmed:** 101,766 rows × 50 columns. Missing values are present in `race`, `medical_specialty`, `payer_code`, and the `_desc` descriptor columns due to unmapped IDs in the source mapping file. No structural issues found.


## Preprocessing Step 2 · Remove Unjustified Duplicate Records
We first check for exact duplicate rows and drop them. Keeping exact duplicates would inflate the model's confidence on specific encounter patterns.


In [ ]:
before = len(df)
df_clean = df.drop_duplicates().copy()
after = len(df_clean)
print(f"Rows before: {before} | Rows after dropping duplicates: {after} | Removed: {before - after}")



**Decision:** 0 exact duplicate rows were found (confirmed by EDA). The `drop_duplicates()` call is kept explicitly to ensure correctness and reproducibility of the pipeline.


## Preprocessing Step 3 · Handle Missing Values
Missing values come in two forms in this dataset:
1. **Placeholder strings** (`?`, `Unknown/Invalid`, `Not Available`) — must be converted to `NaN` so pandas can process them.
2. **Hidden nulls in ID columns** — certain integer IDs (e.g., 5, 6, 8 for admission type) semantically map to "Not Available" and must be explicitly set to `NaN`.

**Strategy:** Informative missingness (e.g., no specialist, no payer) is preserved as an `"Unknown"` category rather than being imputed away.


In [ ]:
import numpy as np

# Step 3a: Replace placeholder strings with NaN
placeholders = ['?', 'Unknown/Invalid', 'Not Available', 'NULL', 'Not Mapped']
df_clean.replace(placeholders, np.nan, inplace=True)

# Step 3b: Drop Hospice / Deceased patients (conceptual leakage — they cannot be readmitted)
dist_before = df_clean['readmitted'].value_counts(normalize=True) * 100
terminal_codes = [11, 13, 14, 19, 20, 21]
df_clean = df_clean[~df_clean['discharge_disposition_id'].isin(terminal_codes)].copy()
dist_after = df_clean['readmitted'].value_counts(normalize=True) * 100
print(f"Shape after removing terminal patients: {df_clean.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
import os
os.makedirs('plots/Preprocessing', exist_ok=True)
sns.barplot(x=dist_before.index, y=dist_before.values, ax=axes[0], palette='Reds', order=['NO', '>30', '<30'])
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.1f%%', fontsize=10)
axes[0].set_title("Target Dist (Before Terminal Drop)")
axes[0].set_ylabel("Percentage (%)")
sns.barplot(x=dist_after.index, y=dist_after.values, ax=axes[1], palette='Greens', order=['NO', '>30', '<30'])
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.1f%%', fontsize=10)
axes[1].set_title("Target Dist (After Terminal Drop)")
plt.savefig('plots/Preprocessing/phase1_terminal_drop.png', bbox_inches='tight')
plt.show()



In [ ]:
# Step 3c: Convert hidden null IDs to NaN before imputation
null_id_codes = {
    'admission_type_id': [5, 6, 8],
    'discharge_disposition_id': [18, 25, 26],
    'admission_source_id': [9, 15, 17, 20, 21]
}
for col, ids in null_id_codes.items():
    df_clean[col] = df_clean[col].replace(ids, np.nan)

# Step 3d: Impute informative missing values — missingness itself carries signal
informative_cols = ['medical_specialty', 'payer_code', 'race',
                    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
                    'admission_type_desc', 'discharge_disposition_desc', 'admission_source_desc']
for col in informative_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

# Step 3e: Impute test results — "None" means "Not tested", which is a valid clinical signal
test_cols = ['max_glu_serum', 'A1Cresult']
for col in test_cols:
    df_clean[col] = df_clean[col].replace('None', 'Not tested').fillna('Not tested')

# Step 3f: Drop a tiny fraction of rows with missing critical fields (gender, primary diagnosis)
rows_before = len(df_clean)
df_clean.dropna(subset=['gender', 'diag_1', 'diag_2', 'diag_3'], inplace=True)
print(f"Rows dropped with missing critical fields (gender/diag): {rows_before - len(df_clean)} (<0.1%)")

print(f"\nShape after full missing value handling: {df_clean.shape}")
print("\nRemaining nulls:")
remaining = df_clean.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else "None — dataset is clean!")



**What we did:**
- Converted `?` / `Unknown/Invalid` placeholder strings to proper `NaN` values.
- Removed ~2,300 terminal patients (Hospice/Expired) to prevent target leakage.
- Converted semantically null integer ID codes to `NaN`.
- Filled informative missing values (`medical_specialty`, `payer_code`, etc.) with `"Unknown"` — preserving the signal that data was absent.
- Filled missing `A1Cresult` / `max_glu_serum` with `"Not tested"` — clinically meaningful.
- Dropped <0.1% of rows with missing primary diagnosis or gender (non-imputable fields).


## Preprocessing Step 4 · Fix Data Types
Several columns are stored as the wrong type. We explicitly cast them to appropriate dtypes to prevent silent errors downstream.


In [ ]:
# ID code columns should be categorical (not numeric) — prevents models from treating them as quantities
code_cols = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in code_cols:
    df_clean[col] = df_clean[col].astype(str)  # Will be mapped to labels in Phase 4 anyway

# Integer columns misread as float due to NaN presence — restore clean int types where applicable
count_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
              'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
for col in count_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(int)

print("Data types after fixing:")
print(df_clean.dtypes.value_counts())
print("\nSample dtypes:")
print(df_clean[code_cols + count_cols].dtypes)



**What we fixed:** ID code columns cast from `int64` → `object` (categorical). Count columns explicitly cast to `int` to remove accidental float representation caused by NaN-induced upcasting during load.


## Preprocessing Step 5 · Handle Inconsistent Data & Invalid Values
We identify and correct values that are syntactically present but semantically invalid or internally inconsistent.


In [ ]:
# Check for 'Unknown/Invalid' gender — filter it out as it is non-imputable
invalid_gender = df_clean[~df_clean['gender'].isin(['Male', 'Female'])]
print(f"Rows with invalid gender values: {len(invalid_gender)}")
df_clean = df_clean[df_clean['gender'].isin(['Male', 'Female'])].copy()

# Verify count columns have no negative values (invalid clinical counts)
for col in count_cols:
    neg = (df_clean[col] < 0).sum()
    if neg > 0:
        print(f"WARNING: {col} has {neg} negative values — clipping to 0")
        df_clean[col] = df_clean[col].clip(lower=0)

# Confirm diag codes — rows where primary diagnosis is literally '?' after NaN conversion may remain
print(f"\nFinal valid gender distribution:")
print(df_clean['gender'].value_counts())
print(f"\nShape after consistency checks: {df_clean.shape}")



**What we handled:** Filtered rows with non-binary gender values (non-imputable). Validated all numerical count columns have no negative values (a medically impossible quantity). Dataset is now internally consistent.


## Preprocessing Step 6 · Transform Columns — Standardize Representations
We apply basic value-level transformations before splitting: converting coded values to meaningful strings and standardizing text representations.


In [ ]:
# Drop the redundant _desc columns — they are exact text equivalents of _id columns
# which will be properly grouped and encoded in Feature Engineering. Keeping them
# would cause double-representation of the same information.
desc_cols_to_drop = ['admission_type_desc', 'discharge_disposition_desc', 'admission_source_desc']
df_clean.drop(columns=desc_cols_to_drop, errors='ignore', inplace=True)
print(f"Dropped redundant descriptor columns: {desc_cols_to_drop}")

# Standardize the 'change' and 'diabetesMed' columns to lowercase for consistency
if 'change' in df_clean.columns:
    df_clean['change'] = df_clean['change'].str.lower().replace({'ch': 'changed', 'no': 'unchanged'})
if 'diabetesMed' in df_clean.columns:
    df_clean['diabetesMed'] = df_clean['diabetesMed'].str.lower()

print(f"\n'change' distribution: {df_clean['change'].value_counts().to_dict()}")
print(f"'diabetesMed' distribution: {df_clean['diabetesMed'].value_counts().to_dict()}")
print(f"\nShape after column transforms: {df_clean.shape}")



**What we transformed:** Dropped the 3 redundant `_desc` columns to prevent double-representation of admission/discharge/source information. Standardized `change` column values from `Ch/No` to human-readable `changed/unchanged`. Standardized `diabetesMed` to lowercase.


## Preprocessing Step 7 · Remove Irrelevant Columns & Prevent Target Leakage
We drop identifier columns (which would cause the model to memorize individual patients) and near-zero variance medication columns (which provide no predictive signal).


In [ ]:
# Drop identifier and constant columns
cols_to_drop = ['weight', 'examide', 'citoglipton', 'encounter_id']
df_clean.drop(columns=cols_to_drop, errors='ignore', inplace=True)
print(f"Dropped identifier / constant columns: {[c for c in cols_to_drop if c not in df_clean.columns]}")

# Dynamically detect and drop near-zero variance medications (>99% 'No')
all_med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
                'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
                'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin',
                'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone',
                'metformin-rosiglitazone', 'metformin-pioglitazone']

dropped_meds = [m for m in all_med_cols if m in df_clean.columns and (df_clean[m] == 'No').mean() > 0.99]
df_clean.drop(columns=dropped_meds, errors='ignore', inplace=True)
print(f"\nDropped {len(dropped_meds)} near-zero variance medications: {dropped_meds}")

# Confirm no target-adjacent columns remain
leakage_risk_cols = ['target']
for col in leakage_risk_cols:
    if col in df_clean.columns:
        df_clean.drop(columns=[col], inplace=True)
        print(f"  Dropped leakage column: {col}")

print(f"\nFinal shape after column removal: {df_clean.shape}")



**What we removed:** Dropped `encounter_id` (pure identifier), `weight` (97% missing), `examide`/`citoglipton` (single-value constants), and dynamically identified near-zero variance medication columns (>99% "No"). Verified no target leakage columns remain in the feature set.


## Preprocessing Step 8 · Investigate & Handle Outliers
We revisit the outlier findings from EDA. Our strategy is justified: outliers in utilization count features (inpatient visits, emergency visits) represent genuine "high-utilizer" patients and are clinically meaningful. We document why we do NOT drop them.


In [ ]:
# Identify numerical count columns for outlier review
count_feats = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
               'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
count_feats = [c for c in count_feats if c in df_clean.columns]

outlier_report = []
for col in count_feats:
    Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_report.append({'Feature': col, 'Q1': Q1, 'Q3': Q3, 'Upper Fence': upper, 'N Outliers': n_outliers, '% of Data': round(n_outliers / len(df_clean) * 100, 2)})

import pandas as pd
outlier_df = pd.DataFrame(outlier_report)
print("Outlier Summary (IQR method):")
from IPython.display import display
display(outlier_df)

print("""
Outlier Strategy: We do NOT drop outliers from utilization features.
These extreme values (e.g., 20+ inpatient visits) represent genuine high-risk
patients — the very population most likely to be readmitted. Removing them would
eliminate the most valuable predictive signal. Instead, we will apply log1p()
transformation in Phase 5 to compress the extreme tails while preserving ordering.
""")



**Decision:** Outliers are **retained** and addressed via `log1p` transformation in the pipeline. Dropping them would remove the highest-risk patients — the exact target population the model is designed to predict. This is a deliberate, justified engineering choice.


## Preprocessing Step 9 · Split the Dataset Using an Appropriate Strategy
We split **by patient ID** (not randomly) to prevent patient memorization leakage. A random split would allow the same patient to appear in both train and test sets across multiple encounters, causing the model to simply memorize the patient's outcome rather than generalize.


In [ ]:
from sklearn.model_selection import train_test_split

# Use only the first encounter per patient to determine stratification label
first_encounters = df_clean.sort_values('patient_nbr').groupby('patient_nbr').first()
patient_targets = first_encounters['readmitted']

train_patients, test_patients = train_test_split(
    patient_targets.index,
    test_size=0.20,
    random_state=42,
    stratify=patient_targets
)

train_df = df_clean[df_clean['patient_nbr'].isin(train_patients)].copy()
test_df  = df_clean[df_clean['patient_nbr'].isin(test_patients)].copy()

print(f"Train set: {train_df.shape} | Test set: {test_df.shape}")
print(f"Train patients: {train_df['patient_nbr'].nunique()} | Test patients: {test_df['patient_nbr'].nunique()}")

# Verify no patient overlap
overlap = set(train_df['patient_nbr']) & set(test_df['patient_nbr'])
print(f"Patient overlap between train and test: {len(overlap)} (must be 0)")

train_dist = train_df['readmitted'].value_counts(normalize=True) * 100
test_dist  = test_df['readmitted'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=train_dist.index, y=train_dist.values, ax=axes[0], palette='Blues', order=['NO', '>30', '<30'])
for c in axes[0].containers: axes[0].bar_label(c, fmt='%.1f%%', fontsize=10)
axes[0].set_title("Train Set Class Distribution")
axes[0].set_ylabel("Percentage (%)")
sns.barplot(x=test_dist.index, y=test_dist.values, ax=axes[1], palette='Oranges', order=['NO', '>30', '<30'])
for c in axes[1].containers: axes[1].bar_label(c, fmt='%.1f%%', fontsize=10)
axes[1].set_title("Test Set Class Distribution")
plt.savefig('plots/Preprocessing/phase2_split_stratification.png', bbox_inches='tight')
plt.show()



**What we verified:** 0 patient overlap between train and test. Class distributions are consistent between both sets (stratified). Train: ~80% of total encounters, Test: ~20%.


# Feature Engineering

All feature engineering below is designed around two core principles:
1. **Train-only learning**: Any mapping derived from the data (top-N categories, etc.) is computed from `train_df` only and applied to `test_df`.
2. **Clinical meaningfulness**: Features are created with domain knowledge, not just mathematical convenience.


## Feature Engineering Step 1 · Feature Creation
We create new aggregate features that capture clinically meaningful signals beyond what individual raw columns express.


In [ ]:
# 1a. Total prior visits (captures overall healthcare utilization intensity)
train_df['total_visits'] = train_df['number_outpatient'] + train_df['number_emergency'] + train_df['number_inpatient']
test_df['total_visits']  = test_df['number_outpatient']  + test_df['number_emergency']  + test_df['number_inpatient']

# 1b. Drug change intensity (how many medications were adjusted this visit)
drug_cols_present = [c for c in train_df.columns if c in
    ['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride','acetohexamide',
     'glipizide','glyburide','tolbutamide','pioglitazone','rosiglitazone','acarbose','miglitol',
     'troglitazone','tolazamide','insulin','glyburide-metformin','glipizide-metformin',
     'glimepiride-pioglitazone','metformin-rosiglitazone','metformin-pioglitazone']]

def count_drug_changes(row): return sum(1 for d in drug_cols_present if row[d] in ['Up', 'Down'])
def count_active_drugs(row): return sum(1 for d in drug_cols_present if row[d] in ['Up', 'Down', 'Steady'])

train_df['num_drug_changes'] = train_df.apply(count_drug_changes, axis=1)
test_df['num_drug_changes']  = test_df.apply(count_drug_changes, axis=1)

train_df['num_active_drugs'] = train_df.apply(count_active_drugs, axis=1)
test_df['num_active_drugs']  = test_df.apply(count_active_drugs, axis=1)

print("New features created: total_visits, num_drug_changes, num_active_drugs")
print(train_df[['total_visits', 'num_drug_changes', 'num_active_drugs']].describe())



**Features created:** `total_visits` (sum of all prior encounter types — a strong predictor of high-utilizer patients), `num_drug_changes` (number of medication adjustments this visit — a proxy for care complexity), `num_active_drugs` (total active medications — a polypharmacy risk signal).


## Feature Engineering Step 2 · Feature Transformation
We apply a log1p transformation to the primary count features to understand the before/after distribution. The actual transformation in the final pipeline is applied in the ColumnTransformer (Step 6) to prevent any test set information leaking into the log scaling.


In [ ]:
# Visualize the transformation effect on the most skewed feature
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_df['number_inpatient'], bins=30, ax=axes[0], color='orange', kde=True)
axes[0].set_title("'number_inpatient' — Raw (Heavily Right-Skewed)")

sns.histplot(np.log1p(train_df['number_inpatient']), bins=30, ax=axes[1], color='purple', kde=True)
axes[1].set_title("'number_inpatient' — After log1p (Compressed)")

plt.tight_layout()
plt.savefig('plots/Preprocessing/fe_log1p_preview.png', bbox_inches='tight')
plt.show()

print("Log1p transformation will be applied inside the ColumnTransformer pipeline to all numeric features.")
print("This ensures no leakage: the same log1p is a fixed mathematical function, not data-dependent.")



**Transformation plan:** `log1p` is a fixed mathematical function (not data-dependent) so it will be applied inside the Scikit-learn pipeline to all numerical features. This safely compresses the heavy right-tail while gracefully handling zeros with `log(1+0) = 0`.


## Feature Engineering Step 3 · Feature Extraction
We extract clinically structured information from the raw ICD-9 diagnosis codes (`diag_1`, `diag_2`, `diag_3`), which contain 700+ unique values. We extract two levels of grouping from this complex column.


In [ ]:
import re

def map_diagnosis(code):
    if pd.isna(code) or str(code) in ['?', 'nan']: return 'Other'
    if str(code).startswith('V') or str(code).startswith('E'): return 'Other'
    try:
        c = float(code)
        if c == 250: return 'Diabetes'
        if 390 <= c <= 459 or c == 785: return 'Circulatory'
        if 460 <= c <= 519 or c == 786: return 'Respiratory'
        if 520 <= c <= 579 or c == 787: return 'Digestive'
        if 800 <= c <= 999: return 'Injury'
        if 710 <= c <= 739: return 'Musculoskeletal'
        if 580 <= c <= 629 or c == 788: return 'Genitourinary'
        if 140 <= c <= 239: return 'Neoplasms'
        return 'Other'
    except: return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    train_df[col] = train_df[col].apply(map_diagnosis)
    test_df[col]  = test_df[col].apply(map_diagnosis)

print("Diagnosis extraction complete. Unique categories (diag_1):")
print(train_df['diag_1'].value_counts())



**What we extracted:** Reduced 700+ raw ICD-9 codes into 9 clinically meaningful categories (`Diabetes`, `Circulatory`, `Respiratory`, etc.) using a rule-based ICD-9 grouper. This vastly reduces dimensionality while preserving clinical signal.


## Feature Engineering Step 4 · Feature Combination
We combine related columns into meaningful intensity ratio features that capture *rate* rather than *volume*, removing the confound of length of stay.


In [ ]:
# Medications per day of hospital stay (treatment intensity)
train_df['meds_per_day'] = train_df['num_medications'] / np.maximum(1, train_df['time_in_hospital'])
test_df['meds_per_day']  = test_df['num_medications']  / np.maximum(1, test_df['time_in_hospital'])

# Lab tests per day of hospital stay (diagnostic intensity)
train_df['lab_tests_per_day'] = train_df['num_lab_procedures'] / np.maximum(1, train_df['time_in_hospital'])
test_df['lab_tests_per_day']  = test_df['num_lab_procedures']  / np.maximum(1, test_df['time_in_hospital'])

print("Combined rate features created: meds_per_day, lab_tests_per_day")
print(train_df[['meds_per_day', 'lab_tests_per_day']].describe())



**Features combined:** `meds_per_day` and `lab_tests_per_day` remove the confound of visit length. A patient with 20 medications in a 2-day stay is very different from one with 20 medications in a 10-day stay. These ratio features capture treatment intensity independently of duration.


## Feature Engineering Step 5 · Feature Selection
We remove the remaining redundant and low-information columns now that we have extracted all the useful signals from them.


In [ ]:
# Drop raw drug columns — we've summarized them into num_drug_changes and num_active_drugs
drugs_to_drop = [d for d in drug_cols_present if d != 'insulin']  # Keep insulin — strongest single-drug predictor
train_df.drop(columns=drugs_to_drop, errors='ignore', inplace=True)
test_df.drop(columns=drugs_to_drop, errors='ignore', inplace=True)
print(f"Dropped {len(drugs_to_drop)} individual drug columns (summarized into aggregate features)")

# Drop patient_nbr — identifier, not a feature
for df_ in [train_df, test_df]:
    df_.drop(columns=['patient_nbr'], errors='ignore', inplace=True)

print(f"\nTrain shape after feature selection: {train_df.shape}")
print(f"Test shape after feature selection:  {test_df.shape}")
print(f"\nRemaining columns:")
print(train_df.columns.tolist())



**What we removed:** All individual drug columns (except `insulin`, the strongest single predictor) replaced by the aggregate `num_drug_changes` and `num_active_drugs`. Dropped `patient_nbr` (identifier — not a feature). Final feature set is lean and non-redundant.


## Feature Engineering Step 6 · Feature Representation Refinement
We apply domain-knowledge-driven grouping to high-cardinality and conceptually clustered categorical columns. Then we precisely route each column to the correct encoder.


In [ ]:
# 6a. Group rare medical specialties — top 10 from TRAIN applied to both
top_specialties = train_df['medical_specialty'].value_counts().nlargest(10).index
train_df['medical_specialty'] = train_df['medical_specialty'].apply(lambda x: x if x in top_specialties else 'Other')
test_df['medical_specialty']  = test_df['medical_specialty'].apply(lambda x: x if x in top_specialties else 'Other')

# 6b. Group rare payer codes — top 10 from TRAIN applied to both
top_payers = train_df['payer_code'].value_counts().nlargest(10).index
train_df['payer_code'] = train_df['payer_code'].apply(lambda x: x if x in top_payers else 'Other')
test_df['payer_code']  = test_df['payer_code'].apply(lambda x: x if x in top_payers else 'Other')

# 6c. Conceptually group admission/discharge/source codes into clinical categories
def map_discharge(v):
    try:
        v = int(float(v))
        if v in [1, 8]: return 'Home'
        if v in [6]: return 'Home_Health'
        if v in [2,3,4,5,10,15,16,17,22,23,24,27,28,29,30]: return 'Facility'
        return 'Other'
    except: return 'Other'

def map_adm_type(v):
    try:
        v = int(float(v))
        if v == 1: return 'Emergency'
        if v == 3: return 'Elective'
        return 'Other'
    except: return 'Other'

def map_adm_source(v):
    try:
        v = int(float(v))
        if v == 7: return 'Emergency_Room'
        if v in [1,2,3]: return 'Referral'
        if v in [4,5,6,10,18,19,22,25,26]: return 'Transfer'
        return 'Other'
    except: return 'Other'

for df_ in [train_df, test_df]:
    df_['discharge_disposition_id'] = df_['discharge_disposition_id'].apply(map_discharge)
    df_['admission_type_id']        = df_['admission_type_id'].apply(map_adm_type)
    df_['admission_source_id']      = df_['admission_source_id'].apply(map_adm_source)

# 6d. Map age from string intervals to ordinal integer (0–9) — treated as numeric
age_map = {'[0-10)':0,'[10-20)':1,'[20-30)':2,'[30-40)':3,'[40-50)':4,
           '[50-60)':5,'[60-70)':6,'[70-80)':7,'[80-90)':8,'[90-100)':9}
train_df['age'] = train_df['age'].map(age_map)
test_df['age']  = test_df['age'].map(age_map)

print("Representation refinement complete!")
print(f"\ndischarge_disposition_id: {train_df['discharge_disposition_id'].unique()}")
print(f"admission_type_id: {train_df['admission_type_id'].unique()}")
print(f"admission_source_id: {train_df['admission_source_id'].unique()}")



**What we refined:** Rare medical specialties and payer codes collapsed to `Other` (preventing overfitting on rare categories with 1–2 patients). Admin codes grouped using clinical definitions from `IDS_mapping.csv`. Age mapped to ordinal `int` (0–9) so it flows through the numeric pipeline (log1p + scaling) rather than being one-hot encoded, which would be wasteful for a truly ordered feature.


## Preprocessing Step 10 · Fit & Apply Encoding, Scaling Using Training Data Only
We assemble a Scikit-learn `ColumnTransformer` that:
- Applies `log1p` + `StandardScaler` to all numerical features
- Applies `OneHotEncoder` (trained on train set) to all categorical features

**Critical:** `fit_transform` is called only on training data. `transform` (no fitting) is called on test data.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Encode target
target_mapping = {'NO': 0, '>30': 1, '<30': 2}
y_train = train_df['readmitted'].map(target_mapping)
y_test  = test_df['readmitted'].map(target_mapping)

X_train = train_df.drop(columns=['readmitted'])
X_test  = test_df.drop(columns=['readmitted'])

# Precisely route each column to the correct transformer
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features   = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

# Build transformers
log_transformer      = FunctionTransformer(np.log1p, validate=True)
numeric_transformer  = Pipeline([('log', log_transformer), ('scaler', StandardScaler())])
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,   numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

# FIT on train only — transform both
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed  = preprocessor.transform(X_test)

cat_features_out  = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numerical_features + list(cat_features_out)

print(f"\nX_train shape before: {X_train.shape} → after: {X_train_preprocessed.shape}")
print(f"X_test  shape before: {X_test.shape} → after: {X_test_preprocessed.shape}")



**What the pipeline does:** Numerical features → `log1p(x)` (skew correction) → `StandardScaler` (zero mean, unit variance). Categorical features → `OneHotEncoder` (fit on train only, `handle_unknown='ignore'` for unseen categories in test). All transformations are strictly trained on the training set only.


## Feature Engineering Step 7 · Feature Validation
Before saving, we perform a final comprehensive validation. This confirms: correctness of final shapes, zero NaN presence, no target/identifier leakage, and prediction-time availability of all features.


In [ ]:
import pandas as pd
from IPython.display import display

X_train_final = pd.DataFrame(X_train_preprocessed, columns=all_feature_names)
X_test_final  = pd.DataFrame(X_test_preprocessed,  columns=all_feature_names)

print("=" * 60)
print("FEATURE VALIDATION REPORT")
print("=" * 60)

# 1. Shape check
print(f"\n[1] Shapes:")
print(f"    X_train: {X_train_final.shape} | y_train: {y_train.shape}")
print(f"    X_test:  {X_test_final.shape}  | y_test:  {y_test.shape}")

# 2. NaN check
train_nans = X_train_final.isnull().sum().sum()
test_nans  = X_test_final.isnull().sum().sum()
print(f"\n[2] Missing values: X_train={train_nans} | X_test={test_nans}")
print(f"    {'✓ PASS — No missing values!' if train_nans == 0 and test_nans == 0 else '✗ FAIL — Missing values present!'}")

# 3. Target leakage check
leakage_keywords = ['readmitted', 'target', 'patient_nbr', 'encounter_id']
leakage_found = [c for c in X_train_final.columns if any(k in c.lower() for k in leakage_keywords)]
print(f"\n[3] Leakage check: {leakage_found if leakage_found else 'None found'}")
print(f"    {'✗ FAIL — Leakage detected!' if leakage_found else '✓ PASS — No target/identifier leakage!'}")

# 4. Target class distribution
print(f"\n[4] Target distribution (y_train):")
print(y_train.value_counts().rename({0:'NO', 1:'>30', 2:'<30'}))

# 5. Feature range sanity (post-scaling all values should be roughly between -5 and +5 for numerics)
numeric_range = X_train_final[numerical_features].abs().max().max()
print(f"\n[5] Max absolute value in numeric features post-scaling: {numeric_range:.2f}")
print(f"    {'✓ PASS — Values in expected range' if numeric_range < 20 else '⚠ WARNING — Values may be outside expected range'}")

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)



**Validation confirmed:**
- ✅ Shapes are consistent between X and y for both train and test sets.
- ✅ Zero NaN values in both final matrices.
- ✅ No target (`readmitted`) or identifier (`patient_nbr`) columns present.
- ✅ Numeric feature values are in expected post-scaling range.
- ✅ All features are derived from information available **at prediction time** (no future leakage).


## Phase 6 · Handle Imbalance & Save Outputs
The dataset has a class imbalance (~11% `<30` readmission). We document our strategy and save all artifacts needed for modeling.


In [ ]:
import joblib
import os

os.makedirs('dataset_processed', exist_ok=True)

# Save final datasets
X_train_final.to_csv('dataset_processed/X_train_final.csv', index=False)
X_test_final.to_csv('dataset_processed/X_test_final.csv', index=False)
y_train.to_csv('dataset_processed/y_train_final.csv', index=False)
y_test.to_csv('dataset_processed/y_test_final.csv', index=False)

# Save the fitted pipeline for inference time
joblib.dump(preprocessor, 'dataset_processed/preprocessing_pipeline.pkl')

print("All preprocessing artifacts saved successfully!")
print("  dataset_processed/X_train_final.csv")
print("  dataset_processed/X_test_final.csv")
print("  dataset_processed/y_train_final.csv")
print("  dataset_processed/y_test_final.csv")
print("  dataset_processed/preprocessing_pipeline.pkl")
print(f"\nClass imbalance strategy: class_weight='balanced' will be applied in all model")
print(f"constructors to algorithmically handle the ~11% minority class (<30 days readmission).")
print(f"SMOTE will be explored during cross-validation experiments.")



**Imbalance Strategy:** We use `class_weight='balanced'` in model constructors rather than downsampling, as downsampling would discard ~45,000 "NO" class rows with legitimate clinical patterns. SMOTE will be explored during cross-validation. The primary evaluation metric is **Recall + Macro F1** on the `<30` class.


# 🏁 Preprocessing & Feature Engineering Handoff Summary
**For the Modeling Team:** The data is now fully cleaned, encoded, and ready for modeling. Here is everything you need to know about what was done and why.

### 1. Final Datasets (`dataset_processed/`)
- `X_train_final.csv` / `y_train_final.csv`
- `X_test_final.csv` / `y_test_final.csv`
- **Shapes:** The final matrices contain precisely aligned columns with 0 missing values.
- **Preprocessing Pipeline:** `preprocessing_pipeline.pkl` contains the fitted `ColumnTransformer`. If you need to transform new unseen data, just load this and call `.transform(new_data)`.

### 2. Splitting Strategy
- **Patient-Level Split:** We did NOT use a random split. We grouped by `patient_nbr` and placed all encounters for a single patient into either train or test. This prevents **patient memorization leakage**.

### 3. Removed Features (And Why)
- `weight`: Dropped (97% missing — imputing creates pure bias).
- `patient_nbr`, `encounter_id`: Dropped (pure identifiers — causes leakage).
- `_desc` columns: Dropped (redundant exact text equivalents of the `_id` columns).
- `chlorpropamide` & 20 other drugs: Dropped (near-zero variance; >99% of patients did not take them).
- **Raw Drug Columns:** Dropped after extracting their signal into the `num_drug_changes` and `num_active_drugs` summaries to prevent severe multicollinearity.

### 4. Selected / Engineered Features
- `total_visits`: Sum of past inpatient, outpatient, and emergency visits. (Strongest proxy for a "high-utilizer" patient).
- `meds_per_day` / `lab_tests_per_day`: Removed the confound of length of stay by creating intensity ratios.
- **Grouped ICD-9 Codes:** Mapped 700+ raw `diag` codes into 9 clinical categories (e.g., 'Diabetes', 'Circulatory') for stability.
- **Grouped Admin Codes:** Mapped ID numbers into semantic categories (e.g., 'Home', 'Facility', 'Emergency').

### 5. Encoding & Scaling
- **Numerical:** Handled outliers via `log1p` transformation (compresses the extreme right tail of high-utilizers without dropping them) + `StandardScaler`.
- **Categorical:** Handled via `OneHotEncoder`. We explicitly chose OHE over Ordinal Encoding because EDA proved the readmission risk does not increase linearly (e.g., a "Down" dose change has a different risk profile than an "Up" change).
- **Age:** Explicitly ordinalized (`[0-10) -> 0`, etc.) and passed through the numeric pipeline.

### 6. Next Steps for Modeling
- **Algorithmic Feature Selection:** We have removed redundant and leaky features. If you want to reduce dimensionality further, apply algorithmic selection (e.g., Lasso L1 penalty, Random Forest feature importance, SelectKBest) as part of your tuning pipeline.
- **Class Imbalance:** The `<30` target class is only ~11%. Use `class_weight='balanced'` in your model constructors (LogisticRegression, RandomForest, etc.) or apply SMOTE to the training set. Evaluate using **Recall** and **Macro F1**.

